<a href="https://colab.research.google.com/github/lilianeuwase2/math_forML-Formative3/blob/main/bayesian_probability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

step 1 : set up

In [2]:
import csv

FILE = "IMDB Dataset.csv"
TARGET = "positive"


step 2 : reading the reviews

In [3]:
def load_reviews(path):
    reviews = []
    with open(path, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            reviews.append((row["review"], row["sentiment"]))
    return reviews


reviews = load_reviews(FILE)
print("Number of reviews:", len(reviews))
print("Label of the first review:", reviews[0][1])

Number of reviews: 5505
Label of the first review: positive


step 3: cleaning

In [4]:
def clean_words(text):
    text = text.lower()
    letters_only = "".join(ch if ch.isalpha() else " " for ch in text)
    words = letters_only.split()
    return set(words)

review_words = [(clean_words(text), label) for text, label in reviews]

# showing how the review is cleaned
first_sentence = reviews[0][0].split(".")[0]
print("Before cleaning:", first_sentence)
print("After cleaning: ", sorted(clean_words(first_sentence)))

Before cleaning: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked
After cleaning:  ['after', 'be', 'episode', 'has', 'hooked', 'just', 'll', 'mentioned', 'of', 'one', 'other', 'oz', 'reviewers', 'that', 'the', 'watching', 'you']


Step 4 :The Prior P counting the positive reviews

In [5]:
TOTAL = len(review_words)
N_POSITIVE = 0

for words, label in review_words:
    if label == TARGET:
        N_POSITIVE += 1

PRIOR = N_POSITIVE / TOTAL
print(f"Total reviews:    {TOTAL}")
print(f"Positive reviews: {N_POSITIVE}")
print(f"Prior P(Positive) = {N_POSITIVE}/{TOTAL} = {PRIOR:.4f}")

Total reviews:    5505
Positive reviews: 2721
Prior P(Positive) = 2721/5505 = 0.4943


Step 5: choosing keywords



we choose the words that clearly express an opinion and are common so that they are able to give reliable numbers

In [6]:
POSITIVE_KEYWORDS = ["excellent", "wonderful", "masterpiece", "loved"]
NEGATIVE_KEYWORDS = ["worst", "waste", "awful", "boring"]
ALL_KEYWORDS = POSITIVE_KEYWORDS + NEGATIVE_KEYWORDS
print("Keywords:", ALL_KEYWORDS)

Keywords: ['excellent', 'wonderful', 'masterpiece', 'loved', 'worst', 'waste', 'awful', 'boring']


step 6 : counting the keywords

---



In [7]:
n_keyword = {}   # how many reviews that has the keyword
n_both = {}      # how many reviews contain the word and are positive
for word in ALL_KEYWORDS:
    n_keyword[word] = 0
    n_both[word] = 0

for words, label in review_words:
    for word in ALL_KEYWORDS:
        if word in words:
            n_keyword[word] += 1
            if label == TARGET:
                n_both[word] += 1

for word in ALL_KEYWORDS:
    print(f"{word:<12} in {n_keyword[word]} reviews, {n_both[word]} of them positive")

excellent    in 397 reviews, 331 of them positive
wonderful    in 338 reviews, 275 of them positive
masterpiece  in 133 reviews, 97 of them positive
loved        in 280 reviews, 221 of them positive
worst        in 473 reviews, 47 of them positive
waste        in 299 reviews, 18 of them positive
awful        in 333 reviews, 30 of them positive
boring       in 343 reviews, 61 of them positive


step 7: Bayes theorem for each keyword

In [8]:
def bayes(word):
    if n_keyword[word] == 0:
        return None
    likelihood = n_both[word] / N_POSITIVE
    marginal = n_keyword[word] / TOTAL
    posterior = (likelihood * PRIOR) / marginal
    return {"keyword": word, "n_keyword": n_keyword[word], "n_both": n_both[word],
            "prior": PRIOR, "likelihood": likelihood,
            "marginal": marginal, "posterior": posterior}


def show_result(r):
    k = r["keyword"]
    print(f"Keyword: '{k}'  (in {r['n_keyword']} reviews, {r['n_both']} of them positive)")
    print(f"  Prior       P(Positive)       = {N_POSITIVE}/{TOTAL} = {r['prior']:.4f}")
    print(f"  Likelihood  P({k} | Positive) = {r['n_both']}/{N_POSITIVE} = {r['likelihood']:.4f}")
    print(f"  Marginal    P({k})            = {r['n_keyword']}/{TOTAL} = {r['marginal']:.4f}")
    print(f"  Posterior   P(Positive | {k}) = {r['likelihood']:.4f} x {r['prior']:.4f} / {r['marginal']:.4f} = {r['posterior']:.4f}")
    print()


results = []
for word in ALL_KEYWORDS:
    r = bayes(word)
    if r is None:
        print(f"'{word}' never appears in the reviews - choose another word")
    else:
        results.append(r)
        show_result(r)

Keyword: 'excellent'  (in 397 reviews, 331 of them positive)
  Prior       P(Positive)       = 2721/5505 = 0.4943
  Likelihood  P(excellent | Positive) = 331/2721 = 0.1216
  Marginal    P(excellent)            = 397/5505 = 0.0721
  Posterior   P(Positive | excellent) = 0.1216 x 0.4943 / 0.0721 = 0.8338

Keyword: 'wonderful'  (in 338 reviews, 275 of them positive)
  Prior       P(Positive)       = 2721/5505 = 0.4943
  Likelihood  P(wonderful | Positive) = 275/2721 = 0.1011
  Marginal    P(wonderful)            = 338/5505 = 0.0614
  Posterior   P(Positive | wonderful) = 0.1011 x 0.4943 / 0.0614 = 0.8136

Keyword: 'masterpiece'  (in 133 reviews, 97 of them positive)
  Prior       P(Positive)       = 2721/5505 = 0.4943
  Likelihood  P(masterpiece | Positive) = 97/2721 = 0.0356
  Marginal    P(masterpiece)            = 133/5505 = 0.0242
  Posterior   P(Positive | masterpiece) = 0.0356 x 0.4943 / 0.0242 = 0.7293

Keyword: 'loved'  (in 280 reviews, 221 of them positive)
  Prior       P(Positi

step 8 : summary table

In [9]:
print("| Keyword | Type | Prior | Likelihood | Marginal | Posterior |")
print("|---|---|---|---|---|---|")
for r in results:
    if r["keyword"] in POSITIVE_KEYWORDS:
        kind = "positive word"
    else:
        kind = "negative word"
    print(f"| {r['keyword']} | {kind} | {r['prior']:.4f} | {r['likelihood']:.4f} | {r['marginal']:.4f} | {r['posterior']:.4f} |")

| Keyword | Type | Prior | Likelihood | Marginal | Posterior |
|---|---|---|---|---|---|
| excellent | positive word | 0.4943 | 0.1216 | 0.0721 | 0.8338 |
| wonderful | positive word | 0.4943 | 0.1011 | 0.0614 | 0.8136 |
| masterpiece | positive word | 0.4943 | 0.0356 | 0.0242 | 0.7293 |
| loved | positive word | 0.4943 | 0.0812 | 0.0509 | 0.7893 |
| worst | negative word | 0.4943 | 0.0173 | 0.0859 | 0.0994 |
| waste | negative word | 0.4943 | 0.0066 | 0.0543 | 0.0602 |
| awful | negative word | 0.4943 | 0.0110 | 0.0605 | 0.0901 |
| boring | negative word | 0.4943 | 0.0224 | 0.0623 | 0.1778 |


step 9: checking that Bayes is correct

In [10]:
for r in results:
    direct = r["n_both"] / r["n_keyword"]
    print(f"{r['keyword']:<12} Bayes = {r['posterior']:.4f}   counting directly = {direct:.4f}")

excellent    Bayes = 0.8338   counting directly = 0.8338
wonderful    Bayes = 0.8136   counting directly = 0.8136
masterpiece  Bayes = 0.7293   counting directly = 0.7293
loved        Bayes = 0.7893   counting directly = 0.7893
worst        Bayes = 0.0994   counting directly = 0.0994
waste        Bayes = 0.0602   counting directly = 0.0602
awful        Bayes = 0.0901   counting directly = 0.0901
boring       Bayes = 0.1778   counting directly = 0.1778
